# FlashBN and ReBN: paper results

Backdoor removal by intervening **only in BatchNorm**; convolutions and the classifier
are never changed.

- **FlashBN** (`bn_defense/flashbn.py`) is training-free. It re-initialises the BN
  statistics, scale/shift and input clipping once from 2,000 clean training images,
  with no gradient step. The defended model processes each test image on its own, in
  eval mode.
- **ReBN** (`bn_defense/rebn.py`) is FlashBN plus retraining of only the BN scale/shift
  on 1,000 clean images for 20 epochs.

**How to run**
1. Put the attacked models in `attacked_models/<model folder>/` (`model.pt` + `poisoned_test.npz`; see
   `attacked_models/README.md`). Models that are missing are skipped.
2. The clean datasets download to `data/` on first use (see `data/README.md`).
3. Run all cells.
   - Finished runs in `results/` are reused when their settings match.
   - `FORCE = True` recomputes everything: 102 runs, about 4 h on 2 GPUs.
   - `RUN = False` only builds the tables.

Metrics: Clean Acc / ASR / RA (%), BackdoorBench protocol.
- **Clean Acc:** the full clean test set.
- **ASR:** every non-target test image with the trigger.
- **RA:** the same poisoned images scored against their true labels.

Each setting runs with seeds 0, 1 and 2. Seed `s` fixes the retraining subset and order
(`seed=s`) and the clean pool (`pool_seed=42+s`).

In [1]:
import json, os, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch

ROOT = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')   # project root
ATTACK_DIR = os.path.join(ROOT, 'attacked_models')
RESULTS_DIR = os.path.join(ROOT, 'results')
GPUS = [f'cuda:{i}' for i in range(torch.cuda.device_count())] or ['cpu']
PARALLEL = 3          # parallel runs (Tiny ImageNet needs ~6 GB each)
SEEDS = [0, 1, 2]
FLASHBN_REFRESH = 'adopted'   # FlashBN statistics refresh after fine-pruning: 'off' | 'adopted' (paper) | 'all'
RUN = True            # False: only build the tables from finished runs
FORCE = False         # True: recompute finished runs
print('project:', ROOT, '| GPUs:', GPUS)

project: /mnt/sda/abka03-data/flashbn_rebn | GPUs: ['cuda:0', 'cuda:1']


## Frozen settings (one per attack)

The values below override `configs/<dataset>.yaml`.
- **FlashBN** uses `hist_clip`.
  - `hist_clip`: exact clean BN statistics, a closed-form BN scale/shift, and clipping of
    every BN input to its clean 0.5–99.5 percentile range.
  - `isc_pct=10` adds per-sample statistic clipping.
  - `fp_ratio` adds BN fine-pruning: the scale/shift of the 5% or 10% least
    clean-active channels in the last 2 BN layers is set to zero. `fp_refresh` then
    re-measures the clean statistics of the BN layer after the pruned one.
  - `FLASHBN_REFRESH` (first code cell) switches the refresh for FlashBN: `'adopted'` (paper:
    Tiny ImageNet Blended, BPP, WaNet), `'off'`, or `'all'` (every attack with fine-pruning).
    Each choice keeps its own result folders.
- **ReBN** uses the best initialisation per attack, then retrains (20 epochs unless set).
  - `pool_size=5000 ft_ratio=…`: more clean data, a 5,000-image pool and 5,000 retraining images
    instead of 2,000 and 1,000. `epochs=50`: longer retraining.
  - Fine-pruning before retraining (`fp_ratio=0.1`) keeps the backdoor from coming back when
    there is more data.
  - `epochs=0` means retraining brought the backdoor back, so ReBN = its step 1.

In [2]:
MODELS = {
    'cifar10': {a: f'cifar10_preactresnet18_{a}_pipeline' for a in
                ['reattack_bb__from_badnet__def_nad', 'ctrl', 'ftrojann', 'bpp', 'poison_ink', 'diffusion_backdoor',
                 'adaptive_blend', 'combat']},
    'tiny': {a: f'tiny_preactresnet18_{a}' for a in
             ['badnet_0_1', 'blended_0_1', 'lf_0_1', 'ssba_0_1', 'wanet_0_1', 'inputaware_0_1', 'bpp_0_1',
              'trojannn_0_1', 'sig_0_005']},
}
HC = dict(init_mode='hist_clip')
ISC = dict(HC, isc_pct=10.0)
GENERAL = dict(ISC, affine_l2=10.0, affine_l2_ref='init')
CONF_OPT = dict(init_mode='confidence_recalib_opt', confidence_patience=2)
PGD = dict(init_mode='pgd_hist_clip', confidence_patience=0)
FP05, FP10 = dict(fp_ratio=0.05), dict(fp_ratio=0.1)
REFRESH = dict(fp_refresh=True)   # re-measure BN statistics after the pruned layer

FLASHBN = {
    'cifar10': {'reattack_bb__from_badnet__def_nad': HC, 'ctrl': ISC, 'ftrojann': ISC, 'bpp': {**ISC, **FP10},
                'poison_ink': ISC, 'diffusion_backdoor': HC, 'adaptive_blend': {**ISC, **FP10},
                'combat': {**ISC, **FP10}},
    'tiny': {'badnet_0_1': HC, 'blended_0_1': {**HC, **FP05, **REFRESH}, 'lf_0_1': {**HC, **FP10},
             'ssba_0_1': {**HC, **FP05}, 'wanet_0_1': {**HC, **FP05, **REFRESH},
             'inputaware_0_1': dict(HC, pool_size=200), 'bpp_0_1': {**HC, **FP05, **REFRESH},
             'trojannn_0_1': HC, 'sig_0_005': HC},
}
# more clean data for ReBN: a 5,000-image pool and 5,000 retraining images (10% of CIFAR-10, 5% of Tiny ImageNet)
MORE = {'cifar10': dict(pool_size=5000, ft_ratio=0.1), 'tiny': dict(pool_size=5000, ft_ratio=0.05)}
E50 = dict(epochs=50)
REBN = {
    'cifar10': {'reattack_bb__from_badnet__def_nad': {**PGD, **MORE['cifar10'], **E50}, 'ctrl': GENERAL,
                'ftrojann': dict(init_mode='fisher_noise'),
                'bpp': dict(init_mode='reset_affine', bn_stats='update', lr=0.01, ft_noise_std=0.05,
                            **MORE['cifar10'], **E50),
                'poison_ink': dict(ISC, epochs=0),
                'diffusion_backdoor': dict(HC, ft_noise_std=0.05, **FP10, **MORE['cifar10']),
                'adaptive_blend': dict(CONF_OPT, l1_gamma=0.0002, **FP10, **MORE['cifar10']),
                'combat': {**ISC, **FP10, 'epochs': 0}},
    'tiny': {'badnet_0_1': {**HC, **MORE['tiny']}, 'blended_0_1': {**CONF_OPT, **FP10, **MORE['tiny']},
             'lf_0_1': CONF_OPT, 'ssba_0_1': {**CONF_OPT, **FP10, **MORE['tiny']},
             'wanet_0_1': {**PGD, **MORE['tiny'], **E50}, 'inputaware_0_1': dict(HC, pool_size=200),
             'bpp_0_1': {**PGD, **MORE['tiny'], **E50}, 'trojannn_0_1': {**HC, **FP10, **REFRESH, **MORE['tiny']},
             'sig_0_005': GENERAL},
}
SETTINGS = {'flashbn': FLASHBN, 'rebn': REBN}
fmt = lambda d: ' '.join(f'{k}={v}' for k, v in d.items())
print(pd.DataFrame([{'dataset': ds, 'attack': a, 'FlashBN': fmt(FLASHBN[ds][a]), 'ReBN': fmt(REBN[ds][a])}
                    for ds in MODELS for a in MODELS[ds]]).set_index(['dataset', 'attack']).to_string())

                                                                                     FlashBN                                                                                                             ReBN
dataset attack                                                                                                                                                                                               
cifar10 reattack_bb__from_badnet__def_nad                                init_mode=hist_clip                              init_mode=pgd_hist_clip confidence_patience=0 pool_size=5000 ft_ratio=0.1 epochs=50
        ctrl                                                init_mode=hist_clip isc_pct=10.0                                               init_mode=hist_clip isc_pct=10.0 affine_l2=10.0 affine_l2_ref=init
        ftrojann                                            init_mode=hist_clip isc_pct=10.0                                                                                    

## Run the defenses (finished runs are reused)

In [3]:
SUFFIX = {'off': '_norefresh', 'adopted': '', 'all': '_refreshall'}[FLASHBN_REFRESH]


def flashbn_setting(setting):
    """Apply FLASHBN_REFRESH to a FlashBN setting (the refresh only acts where fine-pruning is used)."""
    setting = {k: v for k, v in setting.items() if not (FLASHBN_REFRESH == 'off' and k == 'fp_refresh')}
    if FLASHBN_REFRESH == 'all' and setting.get('fp_ratio', 0) > 0:
        setting['fp_refresh'] = True
    return setting


def jobs():
    out = []
    for ds in MODELS:
        for a, folder in MODELS[ds].items():
            for method in SETTINGS:
                setting = flashbn_setting(SETTINGS[method][ds][a]) if method == 'flashbn' else SETTINGS[method][ds][a]
                name = method + (SUFFIX if method == 'flashbn' else '')
                for s in SEEDS:
                    overrides = dict(setting, seed=s, pool_seed=42 + s)
                    out.append(dict(dataset=ds, attack=a, method=method, seed=s, folder=folder, overrides=overrides,
                                    out_dir=os.path.join(RESULTS_DIR, folder, f'{name}_s{s}')))
    return out


def finished(j):
    """Done = metrics.json exists and the run used exactly these settings."""
    try:
        settings = json.load(open(os.path.join(j['out_dir'], 'settings.json')))
        return os.path.exists(os.path.join(j['out_dir'], 'metrics.json')) and settings['overrides'] == j['overrides']
    except (OSError, ValueError):
        return False


def run_one(i_job):
    i, j = i_job
    cmd = [sys.executable, '-m', 'bn_defense.run', j['method'], os.path.join(ATTACK_DIR, j['folder']),
           '--out_dir', j['out_dir'], '--device', GPUS[i % len(GPUS)]]
    for k, v in j['overrides'].items():
        cmd += [f'--{k}', str(v)]
    os.makedirs(j['out_dir'], exist_ok=True)
    t0 = time.time()
    for attempt in range(3):   # retry when the GPU is out of memory
        err = os.path.join(j['out_dir'], 'stderr.txt')
        with open(err, 'w') as f:
            rc = subprocess.run(cmd, cwd=ROOT, stdout=subprocess.DEVNULL, stderr=f).returncode
        if rc == 0 or 'out of memory' not in open(err).read():
            break
        time.sleep(120)
    print(f"[{time.strftime('%H:%M:%S')}] {j['dataset']:7s} {j['attack']:34s} {j['method']:7s} seed {j['seed']} "
          f"{'ok' if rc == 0 else 'FAILED, see ' + err} ({time.time() - t0:.0f}s)", flush=True)
    return rc


ALL = jobs()
present = [j for j in ALL if os.path.exists(os.path.join(ATTACK_DIR, j['folder'], 'model.pt'))]
missing = sorted({j['folder'] for j in ALL} - {j['folder'] for j in present})
if missing:
    print(f'{len(missing)} attacked models not found in {ATTACK_DIR} (skipped):', *missing, sep='\n  ')
todo = [j for j in present if FORCE or not finished(j)]
print(f'{len(ALL)} runs, {len(present)} with a model, {len(todo)} to run')
if RUN and todo:
    todo.sort(key=lambda j: (j['method'] != 'flashbn', j['dataset']))   # fast FlashBN runs first
    with ThreadPoolExecutor(PARALLEL) as ex:
        codes = list(ex.map(run_one, enumerate(todo)))
    print('failed runs:', sum(c != 0 for c in codes))

102 runs, 102 with a model, 0 to run


## Paper tables (Clean Acc / ASR / RA %, mean ± std over seeds)

In [4]:
rows = []
for j in ALL:
    if finished(j):
        m = json.load(open(os.path.join(j['out_dir'], 'metrics.json')))
        rows.append(dict(dataset=j['dataset'], attack=j['attack'], method=j['method'], seed=j['seed'],
                         acc=m['final'][0], asr=m['final'][1], ra=m['final'][2],
                         u_acc=m['original'][0], u_asr=m['original'][1], u_ra=m['original'][2]))
R = pd.DataFrame(rows)
print(f'{len(R)} of {len(ALL)} runs finished')

ms = lambda v: f'{np.mean(v):.1f} ± {np.std(v):.1f}' if len(v) > 1 else f'{np.mean(v):.1f}'
triple = lambda g, p='': f"{ms(g[p + 'acc'])} / {ms(g[p + 'asr'])} / {ms(g[p + 'ra'])}"
NAMES = {'flashbn': 'FlashBN' + {'off': ' (no refresh)', 'adopted': '', 'all': ' (refresh all)'}[FLASHBN_REFRESH],
         'rebn': 'ReBN (FlashBN + retraining)'}
TABLES = {}
for ds in MODELS:
    D = R[R.dataset == ds] if len(R) else R
    if D.empty:
        continue
    out = []
    for a in MODELS[ds]:
        g = D[D.attack == a]
        if g.empty:
            continue
        row = {'attack': a, 'No defense': triple(g.head(1), 'u_')}
        for m, name in NAMES.items():
            gm = g[g.method == m]
            row[name] = triple(gm) if len(gm) else '-'
        out.append(row)
    u = D.drop_duplicates('attack')   # the undefended model does not depend on the seed
    avg = {'attack': 'AVERAGE', 'No defense': f'{u.u_acc.mean():.1f} / {u.u_asr.mean():.1f} / {u.u_ra.mean():.1f}'}
    for m, name in NAMES.items():
        gm = D[D.method == m]
        per_seed = gm.groupby('seed')[['acc', 'asr', 'ra']].mean()   # average over attacks, per seed
        n_ok = (gm.groupby('attack').asr.mean() <= 5).sum()
        avg[name] = f"{triple(per_seed)}  [ASR<=5% on {n_ok}/{gm.attack.nunique()}]" if len(gm) else '-'
    TABLES[ds] = pd.DataFrame(out + [avg]).set_index('attack')
    print(f"\n=== {ds}: Clean Acc / ASR / RA (%), mean ± std over seeds {sorted(D.seed.unique())} ===")
    print(TABLES[ds].to_string())

102 of 102 runs finished

=== cifar10: Clean Acc / ASR / RA (%), mean ± std over seeds [0, 1, 2] ===
                                           No defense                                                FlashBN                            ReBN (FlashBN + retraining)
attack                                                                                                                                                             
reattack_bb__from_badnet__def_nad   81.5 / 7.6 / 78.2                    80.7 ± 0.0 / 5.0 ± 0.0 / 79.3 ± 0.1                    87.6 ± 0.0 / 1.7 ± 0.3 / 87.3 ± 0.2
ctrl                               91.5 / 72.2 / 25.0                    89.1 ± 0.0 / 1.1 ± 0.0 / 82.3 ± 0.2                    90.4 ± 0.1 / 2.4 ± 0.3 / 81.4 ± 0.4
ftrojann                           93.9 / 100.0 / 0.0                    92.4 ± 0.0 / 2.1 ± 0.0 / 83.6 ± 0.2                    93.1 ± 0.1 / 2.7 ± 0.1 / 74.3 ± 1.4
bpp                                 90.1 / 99.8 / 0.2                    87.5 ±